In [12]:
!pip -q install scholarly pandas openpyxl tqdm tenacity

In [13]:
INPUT_XLSX = "pesquisadores.xlsx"   # << seu arquivo
SHEET_NAME = 0                      # primeira aba
COL_NOME   = "Pesquisador_nome"
COL_INST   = "inst_norm"            # usado p/ desambiguar homônimos
COL_ORCID  = None                   # se um dia adicionar, mude para "ORCID"
OUTPUT_XLSX = "pesquisadores_metrics_scholar.xlsx"

REQUESTS_PER_MIN = 20   # ~1 a cada 3s
SLEEP_BASE = 3.0


In [14]:
import unicodedata, re, time
from tqdm import tqdm
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

def norm_text(s: str) -> str:
    if not isinstance(s, str): return ""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def score_candidate(author_dict, full_name_norm: str, inst_norm: str):
    # +2 nome exato; +1 parcial; +3 afiliação bate
    score = 0
    cand_name = norm_text(author_dict.get("name",""))
    if cand_name == full_name_norm:
        score += 2
    elif cand_name and (full_name_norm in cand_name or cand_name in full_name_norm):
        score += 1
    affs = author_dict.get("affiliation","") or author_dict.get("affiliations","")
    if inst_norm and norm_text(affs) and (inst_norm in norm_text(affs) or norm_text(affs) in inst_norm):
        score += 3
    return score

def confidence_from_score(score: int) -> str:
    if score >= 4: return "high"
    if score >= 2: return "medium"
    return "low"


In [15]:
import pandas as pd
from scholarly import scholarly  # se travar por captcha, me avise que trocamos a estratégia

def polite_sleep():
    time.sleep(SLEEP_BASE)

df = pd.read_excel(INPUT_XLSX, sheet_name=SHEET_NAME)

# garante colunas opcionais
if COL_INST not in df.columns: df[COL_INST] = ""
if COL_ORCID and COL_ORCID not in df.columns: df[COL_ORCID] = ""

for c in ["scholar_name","scholar_affiliation","scholar_author_id",
          "citedby","hindex","i10index","pubs_count",
          "match_confidence","match_method","notes"]:
    if c not in df.columns:
        df[c] = None

for i, row in tqdm(df.iterrows(), total=len(df)):
    full_name = str(row[COL_NOME]).strip()
    inst = str(row[COL_INST]).strip() if pd.notna(row[COL_INST]) else ""
    if not full_name:
        continue

    full_name_norm = norm_text(full_name)
    inst_norm = norm_text(inst)

    best_score = -1
    best_cand = None
    note = ""

    try:
        # busca com nome + instituição (quando tiver)
        query = f"{full_name} {inst}" if inst else full_name
        results = list(scholarly.search_author(query))
        polite_sleep()

        # fallback: só nome
        if not results and inst:
            results = list(scholarly.search_author(full_name))
            polite_sleep()

        # escolhe melhor candidato por pontuação
        for cand in results[:10]:
            try:
                basic = scholarly.fill(cand, sections=["basics"])
            except Exception:
                continue
            sc = score_candidate(basic, full_name_norm, inst_norm)
            if sc > best_score:
                best_score, best_cand = sc, basic

        if best_cand:
            summary = scholarly.fill(best_cand, sections=["basics","indices","publications"])
            df.at[i, "scholar_author_id"]   = summary.get("scholar_id")
            df.at[i, "scholar_name"]        = summary.get("name")
            df.at[i, "scholar_affiliation"] = summary.get("affiliation")
            df.at[i, "citedby"]  = summary.get("citedby")
            df.at[i, "hindex"]   = summary.get("hindex")
            df.at[i, "i10index"] = summary.get("i10index")
            pubs = summary.get("publications") or []
            df.at[i, "pubs_count"] = len(pubs)
            df.at[i, "match_confidence"] = confidence_from_score(best_score)
            df.at[i, "match_method"] = "name+inst"
        else:
            df.at[i, "match_confidence"] = "no_match"
            df.at[i, "match_method"] = "name+inst"
            df.at[i, "notes"] = "Sem candidato adequado"

    except Exception as e:
        df.at[i, "match_confidence"] = "error"
        df.at[i, "notes"] = f"{e}"

df.to_excel(OUTPUT_XLSX, index=False)
print("OK ->", OUTPUT_XLSX)


  1%|          | 5/541 [1:11:05<127:00:23, 853.03s/it]


KeyboardInterrupt: 